# 02 — Knowledge Graph Construction

**SONATAM** — *Symbolic Ontology & Neural Audio Transcription Architecture for Music*

This notebook builds the **heterogeneous music knowledge graph** from the
curated feature DataFrame:

1. Load the curated dataset (from NB 01)
2. Build an RDF knowledge graph (rdflib) with the new ontology
3. Export to NetworkX for analytics
4. Convert to **PyTorch Geometric HeteroData** for GraphSAGE training

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
from pathlib import Path

import pandas as pd

from sonata.config.settings import CFG
from sonata.notebook_utils import rp, pp, show_paths  # pp() = absolute path from config string
from sonata.kg import KGBuilder, KGQueries, HarmonicKGSchema as S, HeteroGraphConverter

print("Imports OK ✓")

## 1. Load curated dataset

In [ ]:
parquet_path = pp(CFG["dataset"]["parquet_file"])
print(f"Loading: {rp(parquet_path)}")

df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")
df.head(3)

## 2. Build the RDF knowledge graph

In [ ]:
builder = KGBuilder()
g = builder.from_dataframe(df, include_progressions=False)

print(f"RDF graph: {len(g):,} triples")

In [ ]:
# Save to Turtle
kg_cfg = CFG.get("knowledge_graph", {})
ttl_path = pp(kg_cfg.get("turtle_file", "data/processed/harmonic_kg.ttl"))
builder.save(g, ttl_path)
print(f"✓ Saved RDF → {rp(ttl_path)}")

## 3. SPARQL queries

In [ ]:
q = KGQueries(g)
print(q.summary())

In [ ]:
q.genre_distribution().head(15)

## 4. Convert to PyTorch Geometric HeteroData

This is the input for GraphSAGE training (Notebook 03).

In [ ]:
converter = HeteroGraphConverter(normalize_features=True)
hetero_data = converter.convert(df)

print("HeteroData summary:")
print(hetero_data)
print(f"\nNode types: {hetero_data.node_types}")
print(f"Edge types: {hetero_data.edge_types}")

In [ ]:
# Save HeteroData for training
import torch

hetero_path = pp("data/processed/hetero_data.pt")
hetero_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(hetero_data, hetero_path)
print(f"✓ Saved HeteroData → {rp(hetero_path)}")

## 5. NetworkX visualisation

In [ ]:
import matplotlib.pyplot as plt

G_nx = builder.to_networkx(df.head(100))  # small subset for viz
print(f"NetworkX: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")

node_colors = {
    "MusicalPiece": "#4CAF50",
    "Artist":       "#2196F3",
    "Genre":        "#FF9800",
    "MusicalKey":   "#9C27B0",
    "Era":          "#F44336",
}

colors = [
    node_colors.get(G_nx.nodes[n].get("node_type", ""), "#999")
    for n in G_nx.nodes
]

fig, ax = plt.subplots(figsize=(14, 10))
import networkx as nx
pos = nx.spring_layout(G_nx, seed=42, k=0.5)
nx.draw_networkx(G_nx, pos, ax=ax, node_color=colors, node_size=30,
                 font_size=5, with_labels=False, alpha=0.7)
ax.set_title("Music KG (first 100 pieces)")
plt.tight_layout()
plt.show()